In [1]:
# I want to test other algorithms to. and finding best fitting algorithm for car price prediction
# I am going with multiple linear regression, decision tree, random forest algorithm for same

In [2]:
# importing Libraries

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Importing and reading dataset

ds= pd.read_csv("new_cleaned_data.csv")
ds

,Unnamed: 0,name,company,year,Price,kms_driven,fuel_type
0,62,Indigo eCS LS CR4 BS IV,Tata,2017,200000.0,130000,Diesel
1,121,Suzuki Dzire VDI,Maruti,2015,450000.0,104000,Diesel
2,139,Motors Ambassador,Hindustan,2000,70000.0,200000,Diesel
3,183,Indigo eCS LX TDI BS III,Tata,2016,320000.0,175430,Diesel
4,238,Indigo eCS LX TDI BS III,Tata,2016,320000.0,175400,Diesel
5,303,Scorpio VLX Special Edition BS III,Mahindra,2004,230000.0,160000,Diesel
6,313,Scorpio 2.6 CRDe,Mahindra,2007,220000.0,170000,Diesel
7,379,Suzuki Swift Dzire Tour LDi,Maruti,2016,350000.0,166000,Diesel
8,397,Indigo LX TDI BS III,Tata,2016,130000.0,104000,Diesel
9,417,XUV500 W8,Mahindra,2012,560000.0,100000,Diesel


In [4]:
X=ds[['company','name','year','kms_driven','fuel_type']]
y=ds[['Price']]

In [5]:
# Implementing OneHotcoder

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
ohe=OneHotEncoder()
ohe.fit(X[["company","name","fuel_type"]])

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [6]:
ct=make_column_transformer((OneHotEncoder(handle_unknown = 'ignore',categories=ohe.categories_), ["company","name","fuel_type"]),remainder='passthrough',force_int_remainder_cols=False, sparse_threshold = 0)
ct

,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,False
,categories,"[array(['Chevr... dtype=object), array(['Amaze... dtype=object), ...]"
,drop,None
,sparse_output,True


In [7]:
# making pipeline

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

reg=LinearRegression()
regDec = DecisionTreeRegressor(random_state = 0)
regRFR = RandomForestRegressor(n_estimators =10, random_state= 0)

pipeLinear = make_pipeline(ct, reg)
pipeDec = make_pipeline(ct, regDec)
pipeRFR = make_pipeline(ct, regRFR)

scores = []
for i in range(0,101):
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.10,random_state=i)
    pipeLinear.fit(X_train,y_train)
    pipeDec.fit(X_train,y_train)
    pipeRFR.fit(X_train,y_train)

    result = pipeLinear.predict(X_test)
    scoreLinear = r2_score(y_test, result)
    rmseLinear = np.sqrt(mean_squared_error(y_test, result))
    
    result = pipeDec.predict(X_test)
    scoreDec = r2_score(y_test, result)
    rmseDec = np.sqrt(mean_squared_error(y_test, result))

    result = pipeRFR.predict(X_test)
    scoreRFR = r2_score(y_test, result)
    rmseRFR = np.sqrt(mean_squared_error(y_test, result))
    
    
    scores.append(('Linear',i, scoreLinear,rmseLinear))
    scores.append(('Decision',i, scoreDec, rmseDec ))
    scores.append(('Random Forest',i, scoreRFR, rmseRFR ))

In [8]:
# Finding best r2 score and RMSE score

scoreDF = pd.DataFrame(data = scores, columns = ["algo", "Iteration", "R2 score", "RMSE Score"])
resultDF = scoreDF.sort_values(by="R2 score", ascending = False)
resultDF
# r2 score must be maximum and RMSE score must be minimum

,algo,Iteration,R2 score,RMSE Score
171,Linear,57,0.802442,242056.166489
207,Linear,69,0.793750,238410.542505
175,Decision,58,0.786833,85732.140997
90,Linear,30,0.770196,257455.318250
197,Random Forest,65,0.750367,46226.322522
...,...,...,...,...
141,Linear,47,-45.883480,420640.288755
177,Linear,59,-69.890130,465446.278109
294,Linear,98,-77.978558,263306.219454
33,Linear,11,-88.358654,752446.963560


In [9]:
 # Making Pipeline for best suited algorithm(Decision tree) 

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.10,random_state=79)
pipeDec.fit(X_train,y_train)

,steps,"[('columntransformer', ...), ('decisiontreeregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
# predict for my_input

company=input("enter company name : ")
name=input("enter car name : ")
year=int(input("enter year : "))
kms_driven=int(input("enter kms driven : "))
fuel_type=input("enter fuel type : ")
columns=["company","name","year","kms_driven","fuel_type"]
myinput=pd.DataFrame(columns=columns,data=[[company,name,year,kms_driven,fuel_type]])
result=pipeDec.predict(myinput)
print("You should buy it for ~ price : ",result)

enter company name :  Maruti
enter car name :  Suzuki Swift Dzire Tour LDi
enter year :  2003
enter kms driven :  100000
enter fuel type :  Diesel


You should buy it for ~ price :  [70000.]


In [11]:
# Importing to pkl

import pickle as pkl
pkl.dump(pipeRFR, open("CarProjectUsing_other_algos.pkl","wb"))